# Fine-tuning BETO para NER en Dataset Próstata

**Objetivo**: Entrenar modelo BETO (BERT español) para reconocimiento de entidades nombradas en textos clínicos de cáncer de próstata.

**Modelo**: `dccuchile/bert-base-spanish-wwm-cased` (BETO)

**Dataset**: Próstata (10 tipos de entidades clínicas)

**Configuración**:
- Batch sizes: 8, 16, 32
- Épocas: 6-10 con Early Stopping
- Métricas: F1-score, Precision, Recall
- Objetivo: Superar F1 72.00% (BiLSTM baseline)

---

## 📋 Instrucciones para Kaggle

### 1. Configuración del Notebook
- **Accelerator**: GPU T4 x2
- **Internet**: ON (para descargar modelos y subir a HuggingFace)

### 2. Subir Dataset
Crear carpeta `prostata/` y subir:
- `training.bio`
- `validation_cleaned.bio`
- `testing_cleaned.bio`

### 3. Token de HuggingFace
- Crear token en: https://huggingface.co/settings/tokens
- Permisos: Write
- Usar en celda de login

## 1. Instalación de Librerías

In [152]:
# Instalar librerías (separar evaluate para evitar conflictos)
!pip install transformers==4.30.0 datasets huggingface_hub -q
!pip install evaluate seqeval -q

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  error: subprocess-exited-with-error
  
  × Building wheel for tokenizers (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  ERROR: Failed building wheel for tokenizers
ERROR: ERROR: Failed to build installable wheels for some pyproject.toml based projects (tokenizers)


In [153]:
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")
    print(f"CUDA memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

PyTorch version: 2.10.0+cu128
CUDA available: True
CUDA device: Tesla T4
CUDA memory: 15.64 GB


## 2. Login a HuggingFace

In [ ]:
from huggingface_hub import login

# IMPORTANTE: Reemplazar con tu token de HuggingFace
login("token huggingFace Aquí")

## 3. Configuración

In [155]:
import os
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Desactivar W&B
os.environ["WANDB_DISABLED"] = "true"

# Configuración del modelo
MODEL_CHECKPOINT = "dccuchile/bert-base-spanish-wwm-cased"  # BETO
MAX_LENGTH = 128

# Configuración de entrenamiento
BATCH_SIZE = 16  # Probar con 8, 16, 32
NUM_EPOCHS = 10
LEARNING_RATE = 2e-5

# Rutas del dataset (ajustar según estructura de Kaggle)
DATA_DIR = Path("/kaggle/input/datasets/julianquimbayocastro/prostata")  # o "./prostata" si subes directamente
OUTPUT_DIR = f"./beto-ner-prostata-batch{BATCH_SIZE}"

RUTAS_ARCHIVOS = {
    "train": DATA_DIR / "training.bio",
    "validation": DATA_DIR / "validation_cleaned.bio",
    "test": DATA_DIR / "testing_cleaned.bio"
}

# Nombre del modelo en HuggingFace
HF_MODEL_NAME = "Lucyan85/beto-ner-prostata"

print(f"✅ Configuración:")
print(f"  Modelo base: {MODEL_CHECKPOINT}")
print(f"  Batch size: {BATCH_SIZE}")
print(f"  Épocas máximas: {NUM_EPOCHS}")
print(f"  Output: {OUTPUT_DIR}")
print(f"  HuggingFace: {HF_MODEL_NAME}")

✅ Configuración:
  Modelo base: dccuchile/bert-base-spanish-wwm-cased
  Batch size: 16
  Épocas máximas: 10
  Output: ./beto-ner-prostata-batch16
  HuggingFace: Lucyan85/beto-ner-prostata


## 4. Carga del Dataset

In [156]:
def leer_bio_file(ruta):
    """
    Lee archivo en formato BIO (token\tlabel).
    Retorna diccionario con listas de tokens y etiquetas.
    """
    datos = {"tokens": [], "ner_tags": []}
    tokens, tags = [], []
    
    with open(ruta, encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            
            if not line:
                # Línea vacía = fin de oración
                if tokens:
                    datos["tokens"].append(tokens)
                    datos["ner_tags"].append(tags)
                    tokens, tags = [], []
            else:
                # Línea con datos
                partes = line.split('\t')
                if len(partes) == 2:
                    token, label = partes
                    tokens.append(token)
                    tags.append(label)
        
        # Última oración
        if tokens:
            datos["tokens"].append(tokens)
            datos["ner_tags"].append(tags)
    
    return datos

# Cargar datasets
print("📂 Cargando datasets...")
train_data = leer_bio_file(RUTAS_ARCHIVOS["train"])
val_data = leer_bio_file(RUTAS_ARCHIVOS["validation"])
test_data = leer_bio_file(RUTAS_ARCHIVOS["test"])

print(f"\n✅ Datasets cargados:")
print(f"  Train: {len(train_data['tokens']):,} oraciones")
print(f"  Val: {len(val_data['tokens']):,} oraciones")
print(f"  Test: {len(test_data['tokens']):,} oraciones")

📂 Cargando datasets...

✅ Datasets cargados:
  Train: 3,106 oraciones
  Val: 929 oraciones
  Test: 991 oraciones


## 5. Detección y Mapeo de Etiquetas

In [157]:
def detectar_etiquetas(rutas_dict):
    """
    Detecta todas las etiquetas únicas en los archivos.
    Retorna lista ordenada: etiquetas específicas + 'O' al final.
    """
    todas = set()
    
    for ruta in rutas_dict.values():
        with open(ruta, encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if line and '\t' in line:
                    partes = line.split('\t')
                    if len(partes) == 2:
                        todas.add(partes[1])
    
    # Ordenar: etiquetas específicas alfabéticamente, 'O' al final
    etiquetas = sorted(todas - {"O"}) + ["O"]
    return etiquetas

# Detectar etiquetas
label_list = detectar_etiquetas(RUTAS_ARCHIVOS)
label2id = {label: i for i, label in enumerate(label_list)}
id2label = {i: label for label, i in label2id.items()}

print(f"\n🏷️ Etiquetas detectadas ({len(label_list)}):")
for i, label in enumerate(label_list):
    print(f"  {i}: {label}")

# Guardar mapeo de etiquetas
import json
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
with open(Path(OUTPUT_DIR) / "labels.json", "w") as f:
    json.dump(label_list, f, indent=2)
print(f"\n✅ Etiquetas guardadas en: {OUTPUT_DIR}/labels.json")


🏷️ Etiquetas detectadas (23):
  0: ,
  1: 0
  2: B-BIOMARCADOR
  3: B-CANCER
  4: B-CIRUGIA
  5: B-DOSIS
  6: B-EDAD
  7: B-FECHA
  8: B-GLEASON
  9: B-MEDICAMENTO
  10: B-TNM
  11: B-TRATAMIENTO
  12: I-BIOMARCADOR
  13: I-CANCER
  14: I-CIRUGIA
  15: I-DOSIS
  16: I-EDAD
  17: I-FECHA
  18: I-GLEASON
  19: I-MEDICAMENTO
  20: I-TNM
  21: I-TRATAMIENTO
  22: O

✅ Etiquetas guardadas en: ./beto-ner-prostata-batch16/labels.json


## 6. Creación de Dataset con HuggingFace

In [158]:
from datasets import Dataset, DatasetDict, Features, Sequence, Value, ClassLabel

# Convertir etiquetas de texto a IDs
def convertir_tags_a_ids(datos, label2id):
    datos_ids = {"tokens": [], "ner_tags": []}
    for tokens, tags in zip(datos["tokens"], datos["ner_tags"]):
        datos_ids["tokens"].append(tokens)
        datos_ids["ner_tags"].append([label2id[tag] for tag in tags])
    return datos_ids

train_data_ids = convertir_tags_a_ids(train_data, label2id)
val_data_ids = convertir_tags_a_ids(val_data, label2id)
test_data_ids = convertir_tags_a_ids(test_data, label2id)

# Definir features
features = Features({
    "tokens": Sequence(Value("string")),
    "ner_tags": Sequence(ClassLabel(names=label_list))
})

# Crear datasets
dataset = DatasetDict({
    "train": Dataset.from_dict(train_data_ids, features=features),
    "validation": Dataset.from_dict(val_data_ids, features=features),
    "test": Dataset.from_dict(test_data_ids, features=features)
})

print("\n✅ Dataset creado:")
print(dataset)
print("\n📄 Ejemplo de oración:")
print(f"  Tokens: {dataset['train'][0]['tokens'][:10]}...")
print(f"  Tags: {[id2label[tag] for tag in dataset['train'][0]['ner_tags'][:10]]}...")


✅ Dataset creado:
DatasetDict({
    train: Dataset({
        features: ['tokens', 'ner_tags'],
        num_rows: 3106
    })
    validation: Dataset({
        features: ['tokens', 'ner_tags'],
        num_rows: 929
    })
    test: Dataset({
        features: ['tokens', 'ner_tags'],
        num_rows: 991
    })
})

📄 Ejemplo de oración:
  Tokens: ['Paciente', 'de', '72', 'años', ',', 'con', 'antecedentes', 'médicos', 'de', 'HTA']...
  Tags: ['O', 'O', 'B-EDAD', 'I-EDAD', 'O', 'O', 'O', 'O', 'O', 'O']...


## 7. Tokenización

In [159]:
from transformers import AutoTokenizer

# Cargar tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_CHECKPOINT)
print(f"✅ Tokenizer cargado: {MODEL_CHECKPOINT}")

def tokenize_and_align_labels(examples):
    """
    Tokeniza y alinea las etiquetas con los tokens del tokenizer.
    """
    tokenized_inputs = tokenizer(
        examples["tokens"],
        truncation=True,
        is_split_into_words=True,
        max_length=MAX_LENGTH,
        padding=False  # Padding dinámico con DataCollator
    )

    labels = []
    for i, label in enumerate(examples["ner_tags"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        label_ids = []
        previous_word_idx = None
        
        for word_idx in word_ids:
            if word_idx is None:
                # Tokens especiales ([CLS], [SEP], [PAD])
                label_ids.append(-100)
            elif word_idx != previous_word_idx:
                # Primera subpalabra del token
                label_ids.append(label[word_idx])
            else:
                # Subpalabras siguientes del mismo token
                label_ids.append(-100)
            previous_word_idx = word_idx
        
        labels.append(label_ids)
    
    tokenized_inputs["labels"] = labels
    return tokenized_inputs

# Tokenizar datasets
print("\n🔤 Tokenizando datasets...")
tokenized_datasets = dataset.map(
    tokenize_and_align_labels,
    batched=True,
    remove_columns=dataset["train"].column_names
)

print("✅ Tokenización completada")
print(tokenized_datasets)

✅ Tokenizer cargado: dccuchile/bert-base-spanish-wwm-cased

🔤 Tokenizando datasets...


Map:   0%|          | 0/3106 [00:00<?, ? examples/s]

Map:   0%|          | 0/929 [00:00<?, ? examples/s]

Map:   0%|          | 0/991 [00:00<?, ? examples/s]

✅ Tokenización completada
DatasetDict({
    train: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 3106
    })
    validation: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 929
    })
    test: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 991
    })
})


## 8. Preparación del Modelo

In [160]:
from transformers import AutoModelForTokenClassification, DataCollatorForTokenClassification

# Cargar modelo
model = AutoModelForTokenClassification.from_pretrained(
    MODEL_CHECKPOINT,
    num_labels=len(label_list),
    id2label=id2label,
    label2id=label2id
)

print(f"✅ Modelo cargado: {MODEL_CHECKPOINT}")
print(f"  Parámetros: {sum(p.numel() for p in model.parameters()):,}")
print(f"  Etiquetas: {len(label_list)}")

# Data collator (padding dinámico)
data_collator = DataCollatorForTokenClassification(
    tokenizer=tokenizer,
    padding=True
)

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

BertForTokenClassification LOAD REPORT from: dccuchile/bert-base-spanish-wwm-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
bert.embeddings.position_ids               | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Cons

✅ Modelo cargado: dccuchile/bert-base-spanish-wwm-cased
  Parámetros: 109,277,975
  Etiquetas: 23


## 9. Métricas de Evaluación

In [161]:
import numpy as np

# Intentar cargar evaluate, si falla usar seqeval directamente
try:
    from evaluate import load
    seqeval_metric = load("seqeval")
    use_evaluate = True
    print("✅ Usando evaluate.load('seqeval')")
except (ImportError, Exception) as e:
    # Fallback: usar seqeval directamente
    from seqeval.metrics import f1_score, precision_score, recall_score, accuracy_score
    use_evaluate = False
    print("⚠️ Usando seqeval directamente (sin evaluate)")
    print(f"   Razón: {e}")

def compute_metrics(p):
    """
    Calcula métricas F1, Precision, Recall usando seqeval.
    """
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    # Eliminar tokens ignorados (-100)
    true_predictions = [
        [id2label[p] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]
    true_labels = [
        [id2label[l] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]

    if use_evaluate:
        # Usar evaluate.load("seqeval")
        results = seqeval_metric.compute(predictions=true_predictions, references=true_labels)
        return {
            "precision": results["overall_precision"],
            "recall": results["overall_recall"],
            "f1": results["overall_f1"],
            "accuracy": results["overall_accuracy"]
        }
    else:
        # Fallback: usar seqeval directamente
        return {
            "precision": precision_score(true_labels, true_predictions),
            "recall": recall_score(true_labels, true_predictions),
            "f1": f1_score(true_labels, true_predictions),
            "accuracy": accuracy_score(true_labels, true_predictions)
        }

print("✅ Métricas configuradas")

✅ Usando evaluate.load('seqeval')
✅ Métricas configuradas


## 10. Entrenamiento

In [162]:
from transformers import TrainingArguments, Trainer, EarlyStoppingCallback

# Calcular warmup_steps (10% del total de pasos)
total_steps = (len(tokenized_datasets["train"]) // BATCH_SIZE) * NUM_EPOCHS
warmup_steps = int(0.1 * total_steps)

# Configuración de entrenamiento
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    
    # Entrenamiento
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    learning_rate=LEARNING_RATE,
    
    # Evaluación
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    
    # Optimización
    fp16=torch.cuda.is_available(),
    gradient_accumulation_steps=1,
    warmup_steps=warmup_steps,  # Usar warmup_steps en lugar de warmup_ratio (deprecado)
    weight_decay=0.01,
    
    # Logging
    logging_steps=50,
    report_to="none",
    
    # Guardado
    save_total_limit=2,
    push_to_hub=False
)

# Crear Trainer (sin tokenizer - no es necesario si tenemos data_collator)
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)]
)

print("\n🚀 Iniciando entrenamiento...\n")
print(f"Configuración:")
print(f"  Batch size: {BATCH_SIZE}")
print(f"  Épocas: {NUM_EPOCHS}")
print(f"  Learning rate: {LEARNING_RATE}")
print(f"  Early stopping: 3 épocas sin mejora")
print(f"  FP16: {training_args.fp16}")
print()


🚀 Iniciando entrenamiento...

Configuración:
  Batch size: 16
  Épocas: 10
  Learning rate: 2e-05
  Early stopping: 3 épocas sin mejora
  FP16: True



In [163]:
# Entrenar
train_result = trainer.train()

# Métricas de entrenamiento
metrics = train_result.metrics
trainer.log_metrics("train", metrics)
trainer.save_metrics("train", metrics)

# Guardar modelo y tokenizer
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"\n✅ Modelo y tokenizer guardados en: {OUTPUT_DIR}")

Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,4.695255,0.870889,0.563010,0.404297,0.470633,0.898990
2,0.688836,0.197824,0.888814,0.853516,0.870807,0.976033
3,0.202261,0.072402,0.937621,0.949219,0.943384,0.990676
4,0.093927,0.056463,0.954194,0.962891,0.958522,0.993007
5,0.064737,0.053499,0.971391,0.972656,0.972023,0.994143
6,0.041718,0.047913,0.977214,0.977214,0.977214,0.995398
7,0.035414,0.048771,0.978460,0.975911,0.977184,0.995697
8,0.030007,0.049573,0.979791,0.978516,0.979153,0.995816
9,0.022083,0.046607,0.983660,0.979818,0.981735,0.996235
10,0.019894,0.048025,0.982353,0.978516,0.980431,0.995936


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

***** train metrics *****
  epoch                    =       10.0
  total_flos               =  1695032GF
  train_loss               =     0.4316
  train_runtime            = 0:07:53.81
  train_samples_per_second =     65.553
  train_steps_per_second   =      2.068


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


✅ Modelo y tokenizer guardados en: ./beto-ner-prostata-batch16


## 11. Evaluación en Test

In [164]:
# Evaluar en test
print("\n📊 Evaluando en conjunto de prueba...\n")
test_results = trainer.evaluate(tokenized_datasets["test"])

print("\n🎯 Resultados en Test:")
print(f"  F1-score:  {test_results['eval_f1']*100:.2f}%")
print(f"  Precision: {test_results['eval_precision']*100:.2f}%")
print(f"  Recall:    {test_results['eval_recall']*100:.2f}%")
print(f"  Accuracy:  {test_results['eval_accuracy']*100:.2f}%")

# Guardar resultados
trainer.log_metrics("test", test_results)
trainer.save_metrics("test", test_results)

# Comparación con BiLSTM
bilstm_f1 = 72.00
improvement = test_results['eval_f1']*100 - bilstm_f1
print(f"\n📈 Comparación con BiLSTM (CoNLL-2002):")
print(f"  BiLSTM F1: {bilstm_f1:.2f}%")
print(f"  BETO F1:   {test_results['eval_f1']*100:.2f}%")
print(f"  Diferencia: {improvement:+.2f} puntos")


📊 Evaluando en conjunto de prueba...




🎯 Resultados en Test:
  F1-score:  96.18%
  Precision: 95.86%
  Recall:    96.49%
  Accuracy:  99.35%
***** test metrics *****
  epoch                   =       10.0
  eval_accuracy           =     0.9935
  eval_f1                 =     0.9618
  eval_loss               =     0.0681
  eval_precision          =     0.9586
  eval_recall             =     0.9649
  eval_runtime            = 0:00:04.35
  eval_samples_per_second =    227.675
  eval_steps_per_second   =      7.122

📈 Comparación con BiLSTM (CoNLL-2002):
  BiLSTM F1: 72.00%
  BETO F1:   96.18%
  Diferencia: +24.18 puntos


## 12. Publicar en HuggingFace

In [166]:
# Crear model card
model_card = f"""---
language: es
tags:
- token-classification
- ner
- medical
- spanish
- prostate-cancer
datasets:
- prostata
metrics:
- f1
- precision
- recall
model-index:
- name: BETO NER Próstata
  results:
  - task:
      type: token-classification
      name: Named Entity Recognition
    metrics:
    - type: f1
      value: {test_results['eval_f1']:.4f}
      name: F1-score
    - type: precision
      value: {test_results['eval_precision']:.4f}
      name: Precision
    - type: recall
      value: {test_results['eval_recall']:.4f}
      name: Recall
---

# BETO Fine-tuned for NER on Prostate Cancer Clinical Texts

## Descripción

Modelo BETO (BERT español) afinado para reconocimiento de entidades nombradas (NER) en textos clínicos sobre cáncer de próstata.

**Modelo base**: `dccuchile/bert-base-spanish-wwm-cased`

## Entidades Reconocidas

El modelo identifica 10 tipos de entidades clínicas:

1. **EDAD**: Edad del paciente
2. **BIOMARCADOR**: PSA, marcadores biológicos
3. **CANCER**: Tipos de cáncer
4. **GLEASON**: Score de Gleason
5. **TNM**: Clasificación TNM
6. **TRATAMIENTO**: Tratamientos médicos
7. **MEDICAMENTO**: Medicamentos
8. **DOSIS**: Dosis de medicamentos
9. **CIRUGIA**: Procedimientos quirúrgicos
10. **FECHA**: Fechas de eventos médicos

## Rendimiento

| Métrica | Valor |
|---------|-------|
| F1-score | {test_results['eval_f1']*100:.2f}% |
| Precision | {test_results['eval_precision']*100:.2f}% |
| Recall | {test_results['eval_recall']*100:.2f}% |
| Accuracy | {test_results['eval_accuracy']*100:.2f}% |

## Uso

```python
from transformers import pipeline

# Cargar modelo
ner_pipeline = pipeline(
    "token-classification",
    model="{HF_MODEL_NAME}",
    aggregation_strategy="simple"
)

# Ejemplo
texto = '''Paciente masculino de 72 años con adenocarcinoma de próstata. 
Gleason 3+3, PSA 9.9 ng/dL.'''

resultados = ner_pipeline(texto)
for entidad in resultados:
    print(f"{{entidad['entity_group']}}: {{entidad['word']}}")
```

## Entrenamiento

- **Batch size**: {BATCH_SIZE}
- **Learning rate**: {LEARNING_RATE}
- **Épocas**: {NUM_EPOCHS}
- **Max length**: {MAX_LENGTH}
- **Early stopping**: 3 épocas

## Autor

Lucyan85 - Maestría en Ingeniería de Sistemas y Computación, Universidad Nacional de Colombia
"""

# Guardar README
with open(Path(OUTPUT_DIR) / "README.md", "w", encoding="utf-8") as f:
    f.write(model_card)

print("✅ Model card creado")

✅ Model card creado


In [168]:

# Subir modelo y tokenizer directamente
print("📦 Subiendo modelo...")
model.push_to_hub(
    HF_MODEL_NAME,  # ✅ Primer argumento posicional
    commit_message=f"BETO NER Próstata - F1: {test_results['eval_f1']*100:.2f}% (batch_size={BATCH_SIZE})"
)

print("📦 Subiendo tokenizer...")
tokenizer.push_to_hub(
    HF_MODEL_NAME,
    commit_message=f"BETO NER Próstata - F1: {test_results['eval_f1']*100:.2f}% (batch_size={BATCH_SIZE})"
)

# Subir README (model card)
from huggingface_hub import HfApi, upload_file
api = HfApi()
print("📦 Subiendo README...")
upload_file(
    path_or_fileobj=Path(OUTPUT_DIR) / "README.md",
    path_in_repo="README.md",
    repo_id=HF_MODEL_NAME,
    repo_type="model"
)

📦 Subiendo modelo...


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

📦 Subiendo tokenizer...


README.md: 0.00B [00:00, ?B/s]

📦 Subiendo README...


CommitInfo(commit_url='https://huggingface.co/Lucyan85/beto-ner-prostata/commit/b8cc0685684141e008f862120593de8d6ccaf88d', commit_message='Upload README.md with huggingface_hub', commit_description='', oid='b8cc0685684141e008f862120593de8d6ccaf88d', pr_url=None, repo_url=RepoUrl('https://huggingface.co/Lucyan85/beto-ner-prostata', endpoint='https://huggingface.co', repo_type='model', repo_id='Lucyan85/beto-ner-prostata'), pr_revision=None, pr_num=None)

## 13. Prueba del Modelo Publicado

In [169]:
from transformers import pipeline

# Cargar modelo desde HuggingFace
print(f"\n🧪 Probando modelo publicado: {HF_MODEL_NAME}\n")
ner_pipeline = pipeline(
    "token-classification",
    model=HF_MODEL_NAME,
    aggregation_strategy="simple",
    device=0 if torch.cuda.is_available() else -1
)

# Texto de prueba
texto_prueba = """Paciente masculino de 72 años con antecedentes médicos de hipertensión arterial. 
Actualmente presenta un tumor prostático bilateral confirmado mediante biopsia. 
El diagnóstico histológico reveló un adenocarcinoma de próstata. 
El puntaje de Gleason reportado fue 3+3, lo que indica un cáncer de bajo grado. 
El PSA más alto registrado fue de 9,9 ng/dL."""

print("📄 Texto de prueba:")
print(texto_prueba)
print("\n🔍 Entidades detectadas:\n")

resultados = ner_pipeline(texto_prueba)
for entidad in resultados:
    print(f"  [{entidad['entity_group']}] {entidad['word']} (score: {entidad['score']:.2f})")

print(f"\n✅ Total de entidades detectadas: {len(resultados)}")


🧪 Probando modelo publicado: Lucyan85/beto-ner-prostata



config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/437M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/376 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

📄 Texto de prueba:
Paciente masculino de 72 años con antecedentes médicos de hipertensión arterial. 
Actualmente presenta un tumor prostático bilateral confirmado mediante biopsia. 
El diagnóstico histológico reveló un adenocarcinoma de próstata. 
El puntaje de Gleason reportado fue 3+3, lo que indica un cáncer de bajo grado. 
El PSA más alto registrado fue de 9,9 ng/dL.

🔍 Entidades detectadas:

  [EDAD] 7 (score: 0.99)
  [EDAD] ##2 años (score: 0.95)
  [CANCER] aden (score: 1.00)
  [CANCER] ##ocarcinoma de próstata (score: 0.97)
  [GLEASON] G (score: 1.00)
  [BIOMARCADOR] ##lea (score: 0.41)
  [CANCER] cáncer de bajo grado (score: 0.93)
  [BIOMARCADOR] PSA más alto registrado (score: 0.90)
  [BIOMARCADOR] 9, 9 ng / d (score: 0.89)

✅ Total de entidades detectadas: 9


## 14. Resumen Final

In [170]:
print("="*80)
print("RESUMEN DEL ENTRENAMIENTO")
print("="*80)
print(f"\nModelo: BETO (dccuchile/bert-base-spanish-wwm-cased)")
print(f"Dataset: Próstata (NER clínico)")
print(f"Entidades: {len(label_list)} etiquetas (10 tipos + O)")
print(f"\nConfiguración:")
print(f"  Batch size: {BATCH_SIZE}")
print(f"  Learning rate: {LEARNING_RATE}")
print(f"  Épocas entrenadas: {train_result.metrics.get('epoch', NUM_EPOCHS):.0f}")
print(f"\n🎯 Resultados en Test:")
print(f"  F1-score:  {test_results['eval_f1']*100:.2f}%")
print(f"  Precision: {test_results['eval_precision']*100:.2f}%")
print(f"  Recall:    {test_results['eval_recall']*100:.2f}%")
print(f"\n📊 Comparación:")
print(f"  BiLSTM (CoNLL-2002): {bilstm_f1:.2f}%")
print(f"  BETO (Próstata):     {test_results['eval_f1']*100:.2f}%")
print(f"  Diferencia:          {improvement:+.2f} puntos")
print(f"\n🚀 Modelo publicado: https://huggingface.co/{HF_MODEL_NAME}")
print(f"\n✅ Archivos guardados en: {OUTPUT_DIR}")
print("="*80)

RESUMEN DEL ENTRENAMIENTO

Modelo: BETO (dccuchile/bert-base-spanish-wwm-cased)
Dataset: Próstata (NER clínico)
Entidades: 23 etiquetas (10 tipos + O)

Configuración:
  Batch size: 16
  Learning rate: 2e-05
  Épocas entrenadas: 10

🎯 Resultados en Test:
  F1-score:  96.18%
  Precision: 95.86%
  Recall:    96.49%

📊 Comparación:
  BiLSTM (CoNLL-2002): 72.00%
  BETO (Próstata):     96.18%
  Diferencia:          +24.18 puntos

🚀 Modelo publicado: https://huggingface.co/Lucyan85/beto-ner-prostata

✅ Archivos guardados en: ./beto-ner-prostata-batch16
